In [0]:
%sql
CREATE TABLE IF NOT EXISTS
real_time_catalogue.gold_realtime.fact_sales_unified
USING DELTA
AS
SELECT
    transaction_id,
    transaction_date,
    customer_id,
    product_id,
    quantity,
    unit_price,
    total_amount,
    'batch' AS source_type,
    current_timestamp() AS ingestion_timestamp
FROM real_time_catalogue.gold_realtime.fact_sales_batch;

#####Fusionner les nouvelles transactions Real Time

In [0]:
MERGE INTO
real_time_catalogue.gold_realtime.fact_sales_unified AS target

USING (
    SELECT
        transaction_id,
        transaction_timestamp AS transaction_date,
        customer_id,
        product_id,
        quantity,
        unit_price,
        total_amount,
        'realtime' AS source_type,
        ingestion_timestamp
    FROM real_time_catalogue.silver_realtime.transactions
) AS source

ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
    UPDATE SET
        target.transaction_date = source.transaction_date,
        target.customer_id = source.customer_id,
        target.product_id = source.product_id,
        target.quantity = source.quantity,
        target.unit_price = source.unit_price,
        target.total_amount = source.total_amount,
        target.source_type = source.source_type,
        target.ingestion_timestamp = source.ingestion_timestamp

WHEN NOT MATCHED THEN
    INSERT (
        transaction_id,
        transaction_date,
        customer_id,
        product_id,
        quantity,
        unit_price,
        total_amount,
        source_type,
        ingestion_timestamp
    )
    VALUES (
        source.transaction_id,
        source.transaction_date,
        source.customer_id,
        source.product_id,
        source.quantity,
        source.unit_price,
        source.total_amount,
        source.source_type,
        source.ingestion_timestamp
    );

######Vérifier l’unification

In [0]:
%sql
SELECT
    source_type,
    COUNT(*) AS number_transactions,
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM real_time_catalogue.gold_realtime.fact_sales_unified
GROUP BY source_type;